In [ ]:
from skimage import io, img_as_float
from skimage.morphology import convex_hull_image, remove_small_objects
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import os
import math
%matplotlib inline

inputpath_data = pathlib.Path.cwd() / 'input_data/staining_single_dendrites_morphology' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_data = pathlib.Path.cwd() / 'output_data/staining_single_dendrites_morphology' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/staining_single_dendrites_morphology' 
savepath_fig.mkdir(parents=True, exist_ok=True)

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

palette_genotypes = {'control': '#000000', 'ort_RNAi':'#dd7c2c'}
cm_inch = 1/2.54 
%config InlineBackend.figure_format = 'retina'

In [ ]:
def Vector(x, y, loadpath):
    
    ## Import picture as array
    im = io.imread(loadpath)
    
    ## Set dendritic branching point(dbp) as the maximum value
    im[y, x] = 255
    
    ## Define dendritic branch point (dbp) with the input index (y, x)
    dbp = np.array([y, x])
    
    ## Calculate convex hull area and pixel_num on the same masked image. To make the hull more accurate, remove groups of less than 6 pixels first and then draw the hull. For quantification use all pixels inside hull.
    binary_img = im > np.quantile(im, 0.90)
    denoised = remove_small_objects(binary_img, max_size=5)
    convex_hull = convex_hull_image(denoised)
    convex_hull_diff = img_as_float(convex_hull.copy())
    convex_hull_diff[binary_img] = 2

    pixel_co = np.argwhere(np.logical_and(binary_img, convex_hull))
    pixels_dendrite_in_hull = np.logical_and(binary_img, convex_hull).sum()
    pixels_hull = convex_hull.sum()
    
    ## Subtract the dbp index from every pixel to calculate the vectors
    norm_pixel_values = pixel_co - dbp
    norm_pixel_values[:,0]*= -1
   
    ## Find vector length to every pixel
    vec_length = np.linalg.norm(norm_pixel_values, axis=1)

    ## Delete dbp index
    dbp_index = np.argwhere(vec_length == 0)
    norm_pixel_values = np.delete(norm_pixel_values, (dbp_index), axis=0)
    vec_length = np.delete(vec_length, dbp_index)
    
    ## Normalize vector length 
    norm_vectors = norm_pixel_values / vec_length[:, np.newaxis]

    fig, ax = plt.subplots(1, 3, figsize=(18, 6))

    # 1. Original image
    ax[0].imshow(im, cmap='gray')
    ax[0].scatter(x, y, color='red', marker='x', s=100, label='DBP')
    ax[0].set_title("Original image")
    ax[0].legend()

    # 2. Convex hull
    ax[1].imshow(convex_hull, cmap='gray')
    ax[1].scatter(x, y, color='red', marker='x', s=100)
    ax[1].set_title(f"Convex hull (area = {pixels_hull})")

    # 3. Dendrite pixels inside hull
    overlay = np.zeros((*binary_img.shape, 3))
    overlay[convex_hull] = [0.8, 0.8, 0.8]  # Hull: grey
    overlay[binary_img] = [0, 1, 0]         # Dendrite pixels: green

    ax[2].imshow(overlay)
    ax[2].scatter(x, y, color='red', marker='x', s=100)
    ax[2].set_title(
        f"Dendrite pixels = {pixels_dendrite_in_hull}"
    )

    for a in ax:
        a.axis('off')

    plt.tight_layout()
    plt.show()

    return(norm_vectors, pixels_dendrite_in_hull, pixels_hull)

In [ ]:
data = {}
for genotype in ['control', 'ort_RNAi']:
    for layer in [1,2,3,4]:
        directory = inputpath_data / genotype / ('Layer' + str(layer))
        for file_name in os.listdir(directory):
            if file_name.endswith(".tif"):
                loadpath = directory / file_name
                # savepath = 'Output/' + genotype + '/' +file_name
                coord = os.path.splitext(file_name)
                coord = coord[0].split('_')
                norm_vectors, pixels_dendrite_in_hull, pixels_hull = Vector(int(coord[2]), int(coord[3]), loadpath)
                
                data[file_name] = {'genotype':genotype, 
                                   'layer':layer, 
                                   'file_name':file_name, 
                                   'sin':norm_vectors[:,0],
                                   'cos':norm_vectors[:,1],
                                   'pixels_dendrite_in_hull':pixels_dendrite_in_hull,
                                   'pixels_hull':pixels_hull
                                   }
data = pd.DataFrame(data).T.reset_index(drop=True).set_index(['genotype', 'layer', 'file_name']).reset_index()
data['density'] = data['pixels_dendrite_in_hull'] / data['pixels_hull'] # area is also in px units so resulting density has arbitrary units
data['layer_plotting'] = data.apply(lambda x: x.layer - 0.2 if x.genotype == 'control' else x.layer + 0.2, axis=1)
data['area_micron_sq'] = data.pixels_hull / 211.1209 # scale of images is 14.53 px/µm so 1 µm squared equals 211.1209 px squared

hull_area_means = pd.DataFrame(data.groupby(['genotype','layer']).area_micron_sq.mean()).reset_index()
hull_area_means['layer_plotting'] = hull_area_means.apply(lambda x: x.layer - 0.2 if x.genotype == 'control' else x.layer + 0.2, axis=1)

density_means = pd.DataFrame(data.groupby(['genotype','layer']).density.mean()).reset_index()
density_means['layer_plotting'] = density_means.apply(lambda x: x.layer - 0.2 if x.genotype == 'control' else x.layer + 0.2, axis=1)  

In [ ]:
data['angles'] = data.apply(lambda row: np.arctan2(row.cos, row.sin), axis=1)
data['angles_deg'] = data.apply(lambda row: np.degrees(row.angles), axis=1)

# histogram of all angles of pixels (vectors) in the image
data['hist_bar_length'] = data.apply(lambda row: np.histogram(row.angles_deg, bins = 12, range=(-180, 180))[0], axis=1) 

edges = np.array([-180., -150., -120.,  -90.,  -60.,  -30.,    0.,   30.,   60.,   90.,  120.,  150., 180.])
centers = (edges[:-1] + edges[1:]) / 2

data['hist_bar_centers'] = [centers] * len(data)

data['n_vectors'] = data.apply(lambda row: sum(row.hist_bar_length), axis=1)
data['hist_bar_length_normalized'] = data.apply(lambda row: row.hist_bar_length / row.n_vectors, axis=1)
data

In [ ]:
n_cells = pd.DataFrame(data.groupby(['layer', 'genotype']).file_name.unique())
n_cells['n_cells'] = n_cells.apply(lambda row: len(row.file_name), axis=1)
n_cells = n_cells.reset_index()
n_cells

In [ ]:
plotting_polarplots = data[['genotype', 'layer', 'file_name','hist_bar_centers', 'hist_bar_length_normalized']]
plotting_polarplots = plotting_polarplots.explode(["hist_bar_centers", "hist_bar_length_normalized"],ignore_index=True)
plotting_polarplots

means = pd.DataFrame(plotting_polarplots.groupby(['genotype', 'layer', 'hist_bar_centers']).hist_bar_length_normalized.mean()).rename(columns={'hist_bar_length_normalized':'barlength_mean'})
sem = pd.DataFrame(plotting_polarplots.groupby(['genotype', 'layer', 'hist_bar_centers']).hist_bar_length_normalized.sem()).rename(columns={'hist_bar_length_normalized':'barlength_sem'})
data_polar = pd.concat([means, sem], axis=1).reset_index()
data_polar

In [ ]:
# Orientation index of the dendrites is calculated like Ldir
# Calculate vectors (x and y values) for all histograms for separate cells 
orientation_quantify = data[['genotype', 'layer', 'file_name', 'hist_bar_centers', 'hist_bar_length_normalized']].explode(["hist_bar_centers", "hist_bar_length_normalized"],ignore_index=True)
orientation_quantify['hist_bar_centers'] = orientation_quantify['hist_bar_centers'].astype(float)
orientation_quantify['x'] = orientation_quantify.hist_bar_length_normalized * np.cos(np.deg2rad(orientation_quantify.hist_bar_centers.values))
orientation_quantify['y'] = orientation_quantify.hist_bar_length_normalized * np.sin(np.deg2rad(orientation_quantify.hist_bar_centers.values))
orientation_quantify['single_vec_lengths'] = orientation_quantify.apply(lambda row: math.sqrt(row.x **2 + row.y ** 2), axis=1)

all_x = pd.DataFrame(orientation_quantify.groupby(['file_name']).x.sum())
all_y = pd.DataFrame(orientation_quantify.groupby(['file_name']).y.sum())
all_vectors = all_x.join(all_y).reset_index()

# Resulting vector
all_vectors["res_vec_rad"] = all_vectors.apply(lambda row: math.atan2(row['y'], row['x']), axis=1)
all_vectors["res_vec_deg"] = np.rad2deg(all_vectors["res_vec_rad"])
all_vectors["res_vec_mag"] = all_vectors.apply(lambda row: math.sqrt(row['x'] ** 2 + row['y'] ** 2), axis=1)
sum_subvecs = pd.DataFrame(orientation_quantify.groupby('file_name').single_vec_lengths.sum()).rename(columns={'single_vec_lengths': 'sum_subvecs'}) # It was already normalized so this will be 1

all_vectors = pd.merge(left=all_vectors, right=sum_subvecs, on='file_name')
all_vectors["orientation_index"] = all_vectors["res_vec_mag"] / all_vectors["sum_subvecs"]

orientation_index = orientation_quantify[['genotype', 'layer', 'file_name']].copy().drop_duplicates()
orientation_index = pd.merge(orientation_index, all_vectors[['file_name', 'orientation_index']], on='file_name')

orientation_means = pd.DataFrame(orientation_index.groupby(['genotype','layer']).orientation_index.mean()).reset_index()
orientation_means['layer_plotting'] = orientation_means.apply(lambda x: x.layer - 0.2 if x.genotype == 'control' else x.layer + 0.2, axis=1) 
orientation_means

data_plotting = pd.merge(data, orientation_index[['file_name', 'orientation_index']], on='file_name')

In [ ]:
fig, ax = plt.subplot_mosaic(
      '''
    DDEEFFGG
    HHIIJJKK
    AA.BB.CC
    ''',
    per_subplot_kw={
        "DEFGHIJK": {"projection": "polar"},
    },
    sharex=False,
    sharey=False,
    figsize=(12 * cm_inch, 10 * cm_inch),
    height_ratios=[0.7,0.7,1],
    # gridspec_kw={
    #     "wspace": 4,
    #     "hspace": 0.2}
)

dotsize = 3

sns.scatterplot(data=orientation_means, x='layer_plotting', y='orientation_index', hue='genotype', palette=palette_genotypes, alpha=1, ax=ax['A'], legend=None, zorder=3, marker='_', s=30, lw=1.5)
sns.stripplot(x="layer_plotting", y="orientation_index", hue="genotype", palette=palette_genotypes, data=data_plotting, dodge=False, alpha=0.3, size=dotsize, edgecolor='white', linewidth=0.2, ax=ax['A'], native_scale=True, legend=False, zorder=0)
ax['A'].set_ylim([0,1])
ax['A'].set_ylabel('Orientation index (AU)')

sns.scatterplot(data=hull_area_means, x='layer_plotting', y='area_micron_sq', hue='genotype', palette=palette_genotypes, alpha=1, ax=ax['B'], legend=False, zorder=3, marker='_', s=30, lw=1.5)
sns.stripplot(x="layer_plotting", y="area_micron_sq", hue="genotype", palette=palette_genotypes, data=data_plotting, dodge=False, alpha=0.3, size=dotsize, edgecolor='white', linewidth=0.2, ax=ax['B'], native_scale=True, legend=False, zorder=0)
ax['B'].set_ylim([0,500])
ax['B'].set_ylabel('Dendrite area (µm2)')

sns.scatterplot(data=density_means, x='layer_plotting', y='density', hue='genotype', palette=palette_genotypes, alpha=1, ax=ax['C'], legend=None, zorder=3, marker='_', s=30, lw=1.5)
sns.stripplot(x="layer_plotting", y="density", hue="genotype", palette=palette_genotypes, data=data_plotting, dodge=False, alpha=0.3, size=dotsize, edgecolor='white', linewidth=0.2, ax=ax['C'], native_scale=True, legend=False, zorder=0)
ax['C'].set_ylim([0,0.5])
ax['C'].set_ylabel('Dendrite density (AU)')

for i in 'ABC':
    ax[i].set_xticks([1,2,3,4], labels=['a', 'b', 'c', 'd'])
    ax[i].set_xlabel('T4 subtype')
    ax[i].spines[['top', 'right']].set_visible(False)

axes = ['DEFGHIJK']
layers = [1, 2, 3, 4] * 2
genotypes = ['control'] * 4 + ['ort_RNAi'] * 4
plotting_panels = [{'axis': axis, 'layer': layer, 'genotype': genotype} for axis, layer, genotype in zip(axes[0], layers, genotypes)]
nbins = 12

for condition in plotting_panels:
    axis = condition['axis']
    layer = condition['layer']
    subtype = {1:'a', 2:'b', 3:'c', 4:'d'}[layer]
    genotype = condition['genotype']
    n = n_cells.query("layer == @layer and genotype == @genotype").n_cells.values[0]
    subgroup = data_polar.query("layer == @layer and genotype == @genotype")
    centers = subgroup.hist_bar_centers
    bar_length = subgroup.barlength_mean
    err = subgroup.barlength_sem

    bars = ax[axis].bar(np.radians(centers), bar_length, yerr=err, width = 2*np.pi/nbins - 1, bottom=0.0, alpha = 1, color=palette_genotypes[genotype], lw=0, error_kw={'capsize':1, 'capthick':0.5}, zorder=10)
    ax[axis].set_theta_zero_location("N")  # theta=0 at the top
    ax[axis].set_theta_direction(-1)  # theta increasing counterclockwise
    ax[axis].set_xticks([0, 0.5*np.pi, np.pi, 1.5*np.pi])
    ax[axis].set_xticklabels(['', '', '', ''])
    ax[axis].grid(linewidth=0.5,linestyle='dashed')
    ax[axis].grid(zorder=0)    
    ax[axis].tick_params(axis='x', which='major', pad=-2.5)
    
    ax[axis].set_rlim(-0.1,0.5)
    ax[axis].set_rticks([0, 0.20, 0.40], labels = ['','','']);
    ax[axis].set_rlabel_position(225.5)
    ax[axis].set_title(genotype + ' T4' + str(subtype) + ', n=' + str(n))
ax['D'].set_rticks([0, 0.20, 0.40], labels = ['',20,40]);

figtitle = "morphology_single_cells_dendrite_orientation_area_density.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
data_graphpad_density = data[['genotype', 'layer', 'file_name', 'density']]
data_graphpad_density['genotype_layer'] = data_graphpad_density.apply(lambda row: row.genotype + ' ' + str(row.layer), axis=1)
data_graphpad_density_wide = data_graphpad_density.pivot(index='file_name', columns='genotype_layer', values='density').reset_index().drop(columns='file_name')
data_graphpad_density_wide = data_graphpad_density_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
data_graphpad_density_wide.to_csv(savepath_data / 'morphology_dendrite_density.csv', sep=',', float_format='%.5f')

data_graphpad_area = data[['genotype', 'layer', 'file_name', 'area_micron_sq']]
data_graphpad_area['genotype_layer'] = data_graphpad_area.apply(lambda row: row.genotype + ' ' + str(row.layer), axis=1)
data_graphpad_area_wide = data_graphpad_area.pivot(index='file_name', columns='genotype_layer', values='area_micron_sq').reset_index().drop(columns='file_name')
data_graphpad_area_wide = data_graphpad_area_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
data_graphpad_area_wide.to_csv(savepath_data / 'morphology_dendrite_area_micron_sq.csv', sep=',', float_format='%.5f')

data_graphpad_orientation = orientation_index[['genotype', 'layer', 'file_name', 'orientation_index']]
data_graphpad_orientation['genotype_layer'] = data_graphpad_orientation.apply(lambda row: row.genotype + ' ' + str(row.layer), axis=1)
data_graphpad_orientation_wide = data_graphpad_orientation.pivot(index='file_name', columns='genotype_layer', values='orientation_index').reset_index().drop(columns='file_name')
data_graphpad_orientation_wide = data_graphpad_orientation_wide.apply(lambda x: sorted(x, key=pd.isnull), 0).dropna(how='all')
data_graphpad_orientation_wide.to_csv(savepath_data / 'morphology_dendrite_orientation.csv', sep=',', float_format='%.5f')